Imports:

In [1]:
import aerosandbox as asb
import aerosandbox.numpy as np
from aerosandbox.library import propulsion_electric
import casadi as ca

from wing_deflection import max_deflection

airfoil definitions

In [2]:
import pandas as pd

aero_df = pd.read_csv("naca2412_polar.csv")

alpha_array = aero_df['alpha'].to_numpy()
cl_array = alpha_array * np.pi**2 / 90
cm_array = aero_df['CM'].to_numpy()

# 3. Feed them straight into your CasADi interpolants
CL_func = ca.interpolant("CL_func", "bspline", [alpha_array], cl_array)
CM_func = ca.interpolant("CM_func", "bspline", [alpha_array], cm_array)


Constants, definitions that shouldn't change

In [3]:
wing_airfoil = asb.Airfoil("naca0001")
tail_airfoil = asb.Airfoil("naca0001")

N = 1.2

E = 4e9
E_foam = 23e6

G_boom = E / 15

rho_balsa = 250 # kg/m3
rho_foam = 30

rho = 1.225
nu = 1.5e-5
pi = np.pi



Beginning the optimization environment, initializing some variables


In [4]:
opti = asb.Opti()

V = opti.variable(init_guess = 4, lower_bound = 0.1, upper_bound = 5)
AR = opti.variable(init_guess = 9, lower_bound = 1, upper_bound = 15)
S = opti.variable(init_guess = 0.05, lower_bound = 0.01, upper_bound = 1)
dihedral = opti.variable(init_guess = 5, lower_bound = 0, upper_bound=10)
AR_h = opti.variable(init_guess = 4, lower_bound = 2, upper_bound =10)
AR_v = opti.variable(init_guess = 3, lower_bound = 0.5, upper_bound =5)
S_h = opti.variable(init_guess = 0.01, lower_bound = 0.00001)
S_v = opti.variable(init_guess = 0.005, lower_bound = 0.000001)
H_loc = opti.variable(init_guess = 0.2, lower_bound = 0) #horizontal stabilizer is at H_loc

alpha_normal = opti.variable(init_guess = 3, lower_bound=0, upper_bound = 15)
alpha_banked = opti.variable(init_guess = 5, lower_bound=0, upper_bound = 15)


Derived variables from the given optimized variables

In [5]:
b_total = ca.sqrt(S * AR)
c = S / b_total
b = b_total / 2

b_h = ca.sqrt(S_h * AR_h) / 2
c_h = S_h / (b_h * 2)

b_v = ca.sqrt(S_v * AR_v)
c_v = S_v / b_v

opti.subject_to(c_v == c_h)

MX(fabs(opti0_lam_g_20))

Aerodynamic Constraints

In [6]:
import casadi as ca

COM = opti.variable(init_guess = 0)
weight = opti.variable(init_guess = 0.1, lower_bound = 0)

#tail volume coefficients

l_h = H_loc - c / 4 + c_h / 4
l_v = H_loc - c / 4 + c_v / 4

hor_vol_coef = S_h * l_h / (S * c)
ver_vol_coef = S_v * l_v / (S * c)

opti.subject_to(hor_vol_coef > 0.3)
opti.subject_to(hor_vol_coef < 0.6)

opti.subject_to(ver_vol_coef > 0.02)
opti.subject_to(ver_vol_coef < 0.05)

#aerodynamic constraints

a0 = 2 * pi
e = 0.9
Re = V * c / nu

cl_2d = CL_func(alpha_normal)
cd_2d = 1.328 / ca.sqrt(Re)
cm_2d = 0

cl_2d_bank = CL_func(alpha_banked)
cd_2d_bank = 1.328 / ca.sqrt(Re)
cm_2d_bank = CM_func(alpha_banked)

CL = a0 / (1 + a0 / (pi * AR * e)) * cl_2d
CL_banked = a0 / (1 + a0 / (pi * AR * e)) * cl_2d_bank

#neutral point

# 2. Apply the 3D finite wing correction to get the actual a_w
a_w = a0 / (1 + a0 / (pi * AR * e))
a_h = 2 * pi / (1 + 2 / AR_h)

# np = c * (a_w / (4 * a_h) + hor_vol_coef * (1 + c / (4 * l_h))) / (a_w / a_h + hor_vol_coef * c / l_h )
np = c * (S * a_w / (4 * S_h * a_h) + l_h / c + 0.25) / (S * a_w / (S_h * a_h) + 1)
opti.subject_to(np > COM)

#recalculate neutral point and stuff

q = 1/2 * rho * V**2 * S
q_h = 1/2 * rho * V**2 * S_h

L_normal = q * CL
L_banked = q * CL_banked

#angles of incidence
i_normal = opti.variable(init_guess = 0)
i_banked = opti.variable(init_guess = 0)

downwash = (2 * CL) / (pi * AR)
downwash_banked = (2 * CL_banked) / (pi * AR)
i_alpha_normal = alpha_normal - downwash + i_normal
i_alpha_banked = alpha_banked - downwash_banked + i_banked

CL_h_normal = i_alpha_normal * a_h
CL_h_banked = i_alpha_banked * a_h

L_h_normal = q_h * CL_h_normal
L_h_banked = q_h * CL_h_banked

M_normal = q * cm_2d * c 
M_banked = q * cm_2d_bank * c

CDi_normal = CL ** 2 / (pi * AR * e) + CL_h_normal ** 2 / (pi * AR_h * e)
CDi_banked = CL ** 2 / (pi * AR * e) + CL_h_banked ** 2 / (pi * AR_h * e)
CD0 = cd_2d + 0.04
CD_normal = CDi_normal + CD0
CD_banked = CDi_banked + CD0
D_normal = CD_normal * q
D_banked = CD_banked * q

M_cg_normal = M_normal + L_normal * (COM - (c/4)) - L_h_normal * (H_loc + c_h/4 - COM)
M_cg_banked = M_banked + L_banked * (COM - (c/4)) - L_h_banked * (H_loc + c_h/4 - COM)

opti.subject_to(M_cg_normal == 0)
opti.subject_to(M_cg_banked == 0)

opti.subject_to(CL < 1.4)
opti.subject_to(CL_banked < 1.4)
opti.subject_to(CL_h_normal < 1.4)
opti.subject_to(CL_h_banked < 1.4)

opti.subject_to(L_h_normal + L_normal == weight * 9.8)
opti.subject_to(L_h_banked + L_banked == weight * 9.8 * N)

#spiral stability
spiral_normal = (l_v - c_v / 4) * dihedral / (b_total * CL)
spiral_banked = (l_v - c_v / 4) * dihedral / (b_total * CL_banked)
opti.subject_to(spiral_normal > 5)
opti.subject_to(spiral_banked > 5)

# --- Additional Stability Derivatives ---

# Lift curve slope of the vertical stabilizer 
a_v = 2 * pi / (1 + 2 / AR_v)

# 1. Pitch Stiffness / Longitudinal Stability (Cm_alpha)
deps_dalpha = 2 * a_w / (pi * AR) # Downwash derivative at the tail
CL_alpha_total = a_w + a_h * (S_h / S) * (1 - deps_dalpha)
Cma = -(np - COM) / c * CL_alpha_total

# 2. Weathercock / Directional Stability (Cn_beta)
Cnb = a_v * (S_v / S) * (l_v / b_total)

# 3. Dihedral Effect / Roll Stability (Cl_beta)
# Must be negative. This dictates how strongly the plane levels its wings when it sideslips.
dihedral_rad = dihedral * pi / 180
Clb = - (a_w / 6) * dihedral_rad

# --- Stability Bounds ---
opti.subject_to(Cma < -0.4)   # Forces a decisively stable pitch restoring moment
opti.subject_to(Cnb > 0.04)   # Forces strong yaw restoring moment (weathercocking)
opti.subject_to(Clb < -0.02)  # Ensures the dihedral actually rolls the plane level

opti.subject_to((L_normal / D_normal) > 5)

MX(fabs(opti0_lam_g_40))

Propulsion package

In [7]:
T = D_banked
#mission flight time is 6 mins

energy = 2.96 #Wh
safety_factor = 2
useable_energy = energy * 0.8 / safety_factor

P_avionics = 0.8
P_mech = T * V
powertrain_efficiency = 0.35  # Realistic motor + prop efficiency at micro scales
P_elec = (P_mech / powertrain_efficiency) + P_avionics

time = useable_energy / P_elec

#max thrust
ct0 =  0.0282
ct1 =  -0.0573
ct2 =  -0.2022

Tmax_static = 2              # Maximum thrust desired at static conditions
Rprop       = 0.1016         # Propeller radius (m)
Aprop       = pi*(Rprop**2)         # Propeller disk area

Omega  = ca.sqrt(Tmax_static/(0.5*rho*Rprop**2*Aprop*ct0))

# calculate Lambda, CT, and then Tmax from given variables
Lambda = V/(Omega*Rprop)  # Advance ratio

CT = ct0 + ct1*Lambda + ct2*(Lambda**2) # Thrust coefficient

Tmax = CT*0.5*rho*((Omega*Rprop)**2)*Aprop

opti.subject_to(T < Tmax)


MX(fabs(opti0_lam_g_41))

Structural package

In [8]:
thickness = 0.002 #thin balsa strip
boom_area = 0.01 * 0.01
boom_height = 0.01
margin = 0.01
I_formula = lambda chord, tau: chord * thickness ** 3 / 12

battery_mass = 0.017
motor_mass = 0.03
radio = 0.005
servos = 0.01

# batt_loc = 0
batt_loc = opti.variable(init_guess = 0)
motor_loc  = opti.variable(init_guess = -0.1, upper_bound = 0)

opti.subject_to(batt_loc > motor_loc)

weight_fr = (
    S * thickness * rho_balsa + #wing
    S_h * thickness * rho_balsa +  #hor stab
    S_v * thickness * rho_balsa +  #vert stab
    (H_loc - motor_loc) * rho_balsa * boom_area + #boom
    battery_mass + 
    motor_mass +
    radio + 
    servos + 
    margin
)

cumsum_weight = (
    c / 2 * S * thickness * rho_balsa + #wing
    (H_loc + c_h/2) * S_h * thickness * rho_balsa + #hor stab
    (H_loc + c_v/2) * S_v * thickness * rho_balsa + #vert stab
    (H_loc - motor_loc)**2 / 2 * boom_area * rho_balsa + #boom
    batt_loc * battery_mass + 
    motor_loc * motor_mass +
    servos * H_loc / 2
)

COM_fr = cumsum_weight / weight

opti.subject_to(COM == COM_fr)
opti.subject_to(weight == weight_fr)

# Structural Deflection Constraint (tau set to 0.05 dynamically)
N = 1.2
opti.subject_to(max_deflection(N * weight * 9.81, AR, S, E, I_formula = I_formula, tau = 0.002) < 0.1 * b)
opti.subject_to(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.002) < 0.1 * b_h)
# opti.subject_to(max_deflection(N * weight * 9.81, AR_v, S_v, E, I_formula = I_formula, tau=0.05) < 0.02)

opti.subject_to(H_loc > c)

#torsional considerations
J_boom = 0.141 * (boom_height) ** 4
CL_v_max = 1.0
L_v_max = 0.5 * rho * V**2 * S_v * CL_v_max
torque_boom = L_v_max * (b_v / 2)

twist_angle_rad = (torque_boom * H_loc) / (G_boom * J_boom)
max_twist_deg = 2
max_twist_rad = max_twist_deg * (pi / 180)

opti.subject_to(twist_angle_rad < max_twist_rad)

MX(fabs(opti0_lam_g_49))

Objective function and solving

In [9]:
opti.maximize(time)
# opti.minimize(weight)


# --- Solve the Problem ---
sol = opti.solve(behavior_on_failure="return_last", max_iter=10000)


This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       55
Number of nonzeros in inequality constraint Jacobian.:      114
Number of nonzeros in Lagrangian Hessian.............:       96

Total number of variables............................:       17
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        7
Total number of inequality constraints...............:       42
        inequality constraints with only lower bounds:       18
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       24

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0 -5.8547042e-03 1.87e+01 1.26e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

Output statements

In [10]:
print("\n" + "=" * 45)
print("       AEROSANDBOX DESIGN METRICS")
print("=" * 45)
print(f"Optimization Status : {sol.stats()['return_status']}")
print(f"Total Aircraft Mass : {sol.value(weight) * 1000:.2f} grams")
print(f"Design Velocity     : {sol.value(V):.2f} m/s")

print("-" * 45)
print("MAIN WING")
print(f"Wing Area (S)       : {sol.value(S):.4f} m^2")
print(f"Aspect Ratio (AR)   : {sol.value(AR):.2f}")
print(f"Total Wingspan (b)  : {sol.value(b_total):.2f} m")
print(f"Wing Chord (c)      : {sol.value(c) * 1000:.1f} mm")
print(sol.value(max_deflection(N * weight * 9.81, AR, S, E_foam, tau = c * 0.12) ))
print(f"Dihedral            : {sol.value(dihedral):.1f} deg")

print("-" * 45)
print("TAIL SURFACES & GEOMETRY")
print(f"Horiz. Stabilizer Area (S_h) : {sol.value(S_h):.4f} m^2")
print(f"Horiz. Stabilizer AR (AR_h)  : {sol.value(AR_h):.2f}")
print(f"Horiz. Stabilizer Span (b_h) : {sol.value(b_h * 2):.4f} m")
print(f"Horiz. Stabilizer Chord (c_h): {sol.value(c_h) * 1000:.1f} mm")
print(f"Vert. Stabilizer Area (S_v)  : {sol.value(S_v):.4f} m^2")
print(f"Vert. Stabilizer Span (b_v)  : {sol.value(b_v):.4f} m")
print(f"Vert. Stabilizer Chord (c_v) : {sol.value(c_v) * 1000:.1f} mm")
print(f"Tail Location (H_loc)        : {sol.value(H_loc):.2f} m")

print("-" * 45)
print("AERODYNAMICS & PERF (NORMAL / BANKED)")
print(f"Alpha               : {sol.value(alpha_normal):.2f}° (Normal) | {sol.value(alpha_banked):.2f}° (Banked)")
print(f"Operating CL        : {sol.value(CL):.3f} (Normal) | {sol.value(CL_banked):.3f} (Banked)")
print(f"Total Drag (D)      : {sol.value(D_normal):.3f} N (Normal) | {sol.value(D_banked):.3f} N (Banked)")
print(f"Main Wing Lift      : {sol.value(L_normal):.3f} N (Normal) | {sol.value(L_banked):.3f} N (Banked)")
print(f"Tail Lift           : {sol.value(L_h_normal):.3f} N (Normal) | {sol.value(L_h_banked):.3f} N (Banked)")
print(f"Tail angle          : {sol.value(i_normal):.3f} deg (Normal) | {sol.value(i_banked):.3f} deg (Banked)")

print("-" * 45)
print("STABILITY & BALANCE")
print(f"Center of Mass (COM): {sol.value(COM):.4f} m")
print(f"Neutral Point (NP)  : {sol.value(np):.4f} m")
print(f"Static Margin       : {sol.value((np - COM) / c):.3f} c")
print(f"Horiz. Vol. Coeff.  : {sol.value(hor_vol_coef):.3f} (Target: 0.3 - 0.6)")
print(f"Vert. Vol. Coeff.   : {sol.value(ver_vol_coef):.3f} (Target: 0.02 - 0.05)")
print(f"Spiral Criterion    : {sol.value(spiral_normal):.2f} (Normal) | {sol.value(spiral_banked):.2f} (Banked)")

print("-" * 45)
print("PROPULSION & MASS BREAKDOWN")
print(f"Required Thrust (T) : {sol.value(T):.3f} N")
print(f"Required Thrust (T) : {sol.value(Tmax):.3f} N")
print(f"Energy Required     : {sol.value(energy):.3f} Wh")
print(f"Battery Mass        : {sol.value(battery_mass) * 1000:.1f} g")
print(f"Battery Location    : {sol.value(batt_loc):.3f} m")
print(f"Motor Location      : {sol.value(motor_loc):.3f} m")
print("=" * 45)

print(sol.value(time) * 60)


       AEROSANDBOX DESIGN METRICS
Optimization Status : Solve_Succeeded
Total Aircraft Mass : 121.71 grams
Design Velocity     : 5.00 m/s
---------------------------------------------
MAIN WING
Wing Area (S)       : 0.0697 m^2
Aspect Ratio (AR)   : 3.57
Total Wingspan (b)  : 0.50 m
Wing Chord (c)      : 139.7 mm
0.8986794015457296
Dihedral            : 10.0 deg
---------------------------------------------
TAIL SURFACES & GEOMETRY
Horiz. Stabilizer Area (S_h) : 0.0089 m^2
Horiz. Stabilizer AR (AR_h)  : 10.00
Horiz. Stabilizer Span (b_h) : 0.2982 m
Horiz. Stabilizer Chord (c_h): 29.8 mm
Vert. Stabilizer Area (S_v)  : 0.0015 m^2
Vert. Stabilizer Span (b_v)  : 0.0497 m
Vert. Stabilizer Chord (c_v) : 29.8 mm
Tail Location (H_loc)        : 0.36 m
---------------------------------------------
AERODYNAMICS & PERF (NORMAL / BANKED)
Alpha               : 2.48° (Normal) | 3.03° (Banked)
Operating CL        : 1.053 (Normal) | 1.287 (Banked)
Total Drag (D)      : 0.176 N (Normal) | 0.173 N (Banke

In [11]:
airplane = asb.Airplane(
    name="Optimized Glider",
    xyz_ref=[sol.value(COM), 0, 0],  
    wings=[
        asb.Wing(
            name="Main Wing",
            symmetric=True,  
            xsecs=[  
                asb.WingXSec(  
                    xyz_le=[0, 0, 0],  
                    chord=c,
                    twist=0,  
                    airfoil=wing_airfoil,  
                ),
                asb.WingXSec(  
                    xyz_le=[0, b, b * dihedral * pi / 180],
                    chord=c,
                    twist=0,
                    airfoil=wing_airfoil,
                ),
            ],
        ),
        asb.Wing(
            name="Horizontal Stabilizer",
            symmetric=True,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_h,
                    twist=i_normal,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, b_h, 0], chord=c_h, twist=i_normal, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
        asb.Wing(
            name="Vertical Stabilizer",
            symmetric=False,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_v,
                    twist=0,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, 0, b_v], chord=c_v, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
    ],
)

optimized_airplane = sol.value(airplane)
optimized_airplane.draw()

Widget(value='<iframe src="http://localhost:39109/index.html?ui=P_0x7a86db3449e0_0&reconnect=auto" class="pyvi…

PolyData (0x7a86db32dae0)
  N Cells:    725
  N Points:   730
  N Strips:   0
  X Bounds:   0.000e+00, 3.858e-01
  Y Bounds:   -2.495e-01, 2.495e-01
  Z Bounds:   -6.881e-04, 4.970e-02
  N Arrays:   0

In [ ]:
opti.debug.show_infeasibilities()


